In [2]:
from __future__ import annotations

import argparse
import os
import random
import re
from typing import Dict, List

import pandas as pd

from openai import OpenAI

In [3]:
EXCEL_PATH = r"Prompts.xlsx"
SHEET_NAME = "test"
OUTPUT_PATH = "survey_results.xlsx"

In [ ]:
MODEL = "gpt-4o"
TEMPERATURE = 0.7
ITERATIONS = 10

SEED = 123  # set to None for fully random shuffles

api_key = "XXX"

In [4]:
client = OpenAI(api_key=api_key)
response = client.responses.create(
    model="gpt-5",
    input="Write a haiku about Python programming."
)
print(response.output_text)

Indentation’s grace
Snakes through logic, clear and calm
Tests shed skins of bugs


In [8]:
# =========================
# CONFIG
# =========================
VERSIONS = [
    "BaselineVersion",
    "PrimedVersion",
    "BizarreVersion",
    "ChangedVersion",
    "ConfirmationVersion",
    "FlawsVersion",
]

TASKS = ["Credibility", "ImportancePerso", "ImportanceSoc", "Sharing"]

REQUIRED_COLS = ["Order", "ProfilingPrompt", *VERSIONS, *TASKS]


# =========================
# HELPERS
# =========================
def _clean_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()


def parse_six_brackets(text):
    items = re.findall(r"\[(.*?)\]", text, flags=re.DOTALL)
    items = [i.strip() for i in items if i.strip()]
    if len(items) != 6:
        raise ValueError(f"Expected 6 bracket items, got {len(items)}")
    return [f"[{i}]" for i in items]


def build_initial_prompt(row, shown_versions):
    profiling_prompt = _clean_cell(row["ProfilingPrompt"])

    texts_block = "\n\n".join(
        [f"TEXT {i+1}:\n{_clean_cell(row[v])}" for i, v in enumerate(shown_versions)]
    )

    return f"{profiling_prompt}\n\n{texts_block}"

In [20]:
# =========================
# MAIN RUNNER
# =========================
if SEED is not None:
    random.seed(SEED)

client = OpenAI(api_key=api_key)

df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in Excel: {missing}")

all_results = []

for _, row in df.iterrows():

    order_value = _clean_cell(row["Order"])

    for iteration in range(1, ITERATIONS + 1):

        print("\n" + "="*80)
        print(f"Order={order_value} | Iteration={iteration}")
        print("="*80)

        # Randomize version order
        shown_versions = VERSIONS[:]
        random.shuffle(shown_versions)

        print("Version order:", shown_versions)

        # Start conversation for this iteration
        messages = [
            {
                "role": "user",
                "content": build_initial_prompt(row, shown_versions)
            }
        ]

        # Randomize task order for this iteration
        task_order = TASKS[:]
        random.shuffle(task_order)

        print("Task order:", task_order)

        # Ask tasks sequentially
        for task in task_order:

            task_prompt = _clean_cell(row[task])

            print("\n--- Asking Task:", task, "---")

            messages.append(
                {"role": "user", "content": task_prompt}
            )

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=TEMPERATURE,
            )

            answer = response.choices[0].message.content.strip()

            print("Response received.")

            # Append assistant response to conversation history
            messages.append(
                {"role": "assistant", "content": answer}
            )

            # Parse ranking
            try:
                ranks = parse_six_brackets(answer)
            except Exception:
                ranks = [answer, "", "", "", "", ""]

            all_results.append(
                {
                    "Order": order_value,
                    "task": task,
                    "iteration": iteration,
                    "rank_1": ranks[0],
                    "rank_2": ranks[1],
                    "rank_3": ranks[2],
                    "rank_4": ranks[3],
                    "rank_5": ranks[4],
                    "rank_6": ranks[5],
                }
            )

# Save final results
results_df = pd.DataFrame(
    all_results,
    columns=["Order", "task", "iteration", "rank_1", "rank_2", "rank_3", "rank_4", "rank_5", "rank_6"],
)

results_df.to_excel(OUTPUT_PATH, index=False)

print("\nSaved to:", OUTPUT_PATH)
results_df.head()


Order=virus - default | Iteration=1
Version order: ['ChangedVersion', 'ConfirmationVersion', 'PrimedVersion', 'FlawsVersion', 'BizarreVersion', 'BaselineVersion']
Task order: ['ImportanceSoc', 'ImportancePerso', 'Sharing', 'Credibility']

--- Asking Task: ImportanceSoc ---
Response received.

--- Asking Task: ImportancePerso ---
Response received.

--- Asking Task: Sharing ---
Response received.

--- Asking Task: Credibility ---
Response received.

Order=virus - default | Iteration=2
Version order: ['ChangedVersion', 'BaselineVersion', 'PrimedVersion', 'BizarreVersion', 'FlawsVersion', 'ConfirmationVersion']
Task order: ['ImportanceSoc', 'Sharing', 'Credibility', 'ImportancePerso']

--- Asking Task: ImportanceSoc ---
Response received.

--- Asking Task: Sharing ---
Response received.

--- Asking Task: Credibility ---
Response received.

--- Asking Task: ImportancePerso ---
Response received.

Order=virus - default | Iteration=3
Version order: ['ChangedVersion', 'FlawsVersion', 'Baseli

,Order,task,iteration,rank_1,rank_2,rank_3,rank_4,rank_5,rank_6
0,virus - default,ImportanceSoc,1,[New respiratory virus variant: What we know s...,[New respiratory virus variant: How it connect...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: Are people und...,[New respiratory virus variant: Young mother a...,[New respiratory virus variant: You may alread...
1,virus - default,ImportancePerso,1,[New respiratory virus variant: Young mother a...,[New respiratory virus variant: You may alread...,[New respiratory virus variant: Are people und...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: How it connect...,[New respiratory virus variant: What we know s...
2,virus - default,Sharing,1,[New respiratory virus variant: What we know s...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: How it connect...,[New respiratory virus variant: Are people und...,[New respiratory virus variant: You may alread...,[New respiratory virus variant: Young mother a...
3,virus - default,Credibility,1,[New respiratory virus variant: What we know s...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: How it connect...,[New respiratory virus variant: Are people und...,[New respiratory virus variant: You may alread...,[New respiratory virus variant: Young mother a...
4,virus - default,ImportanceSoc,2,[New respiratory virus variant: Young mother a...,[New respiratory virus variant: Are people und...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: How it connect...,[New respiratory virus variant: What we know s...,[New respiratory virus variant: You may alread...


In [ ]:
##############
# same with Ollama
##############

In [12]:
from ollama import chat
from ollama import ChatResponse

MODEL = "llama3.1:8b"

# =========================
# MAIN RUNNER
# =========================
if SEED is not None:
    random.seed(SEED)

df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in Excel: {missing}")

all_results = []

for _, row in df.iterrows():

    order_value = _clean_cell(row["Order"])

    for iteration in range(1, ITERATIONS + 1):

        print("\n" + "=" * 80)
        print(f"Order={order_value} | Iteration={iteration}")
        print("=" * 80)

        # Randomize version order
        shown_versions = VERSIONS[:]
        random.shuffle(shown_versions)

        print("Version order:", shown_versions)

        # Save the exact shown order as text
        shown_versions_text = " | ".join(shown_versions)

        # Start conversation for this iteration
        messages = [
            {
                "role": "system",
                "content": (
                    "Return exactly six bracketed items with: "
                    "[FULL TITLE (not text number); JUSTIFICATION]. NOTHING ELSE. "
                    "Do not add any introduction, explanation, or concluding sentence."
                )
            },
            {
                "role": "user",
                "content": build_initial_prompt(row, shown_versions)
            }
        ]

        # Randomize task order for this iteration
        task_order = TASKS[:]
        random.shuffle(task_order)

        print("Task order:", task_order)

        # Ask tasks sequentially
        for task in task_order:

            task_prompt = _clean_cell(row[task])

            print("\n--- Asking Task:", task, "---")

            messages.append({"role": "user", "content": task_prompt})

            response: ChatResponse = chat(
                model=MODEL,
                messages=messages,
                options={"temperature": TEMPERATURE}
            )

            answer = response.message.content.strip()

            print("Response received.")

            # Append assistant response to conversation history
            messages.append({"role": "assistant", "content": answer})

            # Parse ranking
            try:
                ranks = parse_six_brackets(answer)
            except Exception:
                ranks = [answer, "", "", "", "", ""]

            all_results.append(
                {
                    "Order": order_value,
                    "task": task,
                    "iteration": iteration,

                    # save the full shown order in one column
                    "shown_versions_order": shown_versions_text,

                    # optional: save each shown title separately too
                    "shown_title_1": shown_versions[0] if len(shown_versions) > 0 else "",
                    "shown_title_2": shown_versions[1] if len(shown_versions) > 1 else "",
                    "shown_title_3": shown_versions[2] if len(shown_versions) > 2 else "",
                    "shown_title_4": shown_versions[3] if len(shown_versions) > 3 else "",
                    "shown_title_5": shown_versions[4] if len(shown_versions) > 4 else "",
                    "shown_title_6": shown_versions[5] if len(shown_versions) > 5 else "",

                    "raw_response": answer,

                    "rank_1": ranks[0],
                    "rank_2": ranks[1],
                    "rank_3": ranks[2],
                    "rank_4": ranks[3],
                    "rank_5": ranks[4],
                    "rank_6": ranks[5],
                }
            )

# Save final results
results_df = pd.DataFrame(
    all_results,
    columns=[
        "Order",
        "task",
        "iteration",
        "shown_versions_order",
        "shown_title_1",
        "shown_title_2",
        "shown_title_3",
        "shown_title_4",
        "shown_title_5",
        "shown_title_6",
        "raw_response",
        "rank_1",
        "rank_2",
        "rank_3",
        "rank_4",
        "rank_5",
        "rank_6",
    ],
)

results_df.to_excel(OUTPUT_PATH, index=False)

print("\nSaved to:", OUTPUT_PATH)
print(results_df.head())


Order=virus - default | Iteration=1
Version order: ['ChangedVersion', 'ConfirmationVersion', 'PrimedVersion', 'FlawsVersion', 'BizarreVersion', 'BaselineVersion']
Task order: ['ImportanceSoc', 'ImportancePerso', 'Sharing', 'Credibility']

--- Asking Task: ImportanceSoc ---
Response received.

--- Asking Task: ImportancePerso ---
Response received.

--- Asking Task: Sharing ---
Response received.

--- Asking Task: Credibility ---
Response received.

Order=virus - default | Iteration=2
Version order: ['ChangedVersion', 'BaselineVersion', 'PrimedVersion', 'BizarreVersion', 'FlawsVersion', 'ConfirmationVersion']
Task order: ['ImportanceSoc', 'Sharing', 'Credibility', 'ImportancePerso']

--- Asking Task: ImportanceSoc ---
Response received.

--- Asking Task: Sharing ---
Response received.

--- Asking Task: Credibility ---
Response received.

--- Asking Task: ImportancePerso ---
Response received.

Order=virus - default | Iteration=3
Version order: ['ChangedVersion', 'FlawsVersion', 'Baseli